In [ ]:
!pip install -q -U "google-genai>=2.3.0"

# 🐾 Clyvo - Assistente Virtual de Saúde Pet

> **Bem-vindo ao projeto Challenge da Clyvo!**

Nosso objetivo é desenvolver um assistente virtual inovador para o segmento de saúde pet, combinando:

- **Base:** Um Chatbot Conversacional Interativo, para uma comunicação fluida e eficiente com os tutores.
- **Features Adicionais:**
  - **Recomendação Algorítmica de Planos:** Um sistema inteligente que sugere o plano de saúde ideal com base nas necessidades do pet e orçamento do tutor.
  - **Geolocalização de Clínicas Parceiras:** Ferramenta para localizar rapidamente clínicas e hospitais credenciados, especialmente em emergências.

---

### 📌 Arquitetura da Solução: A Base Inteligente da Clyvo

Para garantir um atendimento de excelência e segurança, nossa arquitetura é composta por:

- **System Prompt & Guardrails:** Define a personalidade acolhedora da Clyvo, estabelece regras claras de atendimento e protege contra interações fora do escopo, mantendo o foco na saúde e bem-estar dos pets.
- **Histórico Conversacional:** Mantém o contexto de cada diálogo em tempo real, permitindo que a Clyvo 'lembre' das informações do pet e do tutor para um atendimento personalizado e contínuo.
- **Features / Tools:** As ferramentas que tornam a Clyvo superpoderosa!
  - `recomendar_plano`: Nosso motor de regras avançado que, com base em dados do pet e do tutor, recomenda o plano perfeito, sempre respeitando o orçamento e as necessidades específicas.
  - `buscar_clinicas_proximas`: Um algoritmo de Haversine otimizado para localizar com precisão emergências e a rede de clínicas parceiras da Clyvo, garantindo que a ajuda esteja sempre ao alcance.

## 1. 🤖 Configurando o Cliente da API Gemini

Nesta seção, inicializamos o cliente da API do Google Gemini. Isso nos permite interagir com o modelo de linguagem, enviando prompts e recebendo respostas para alimentar a inteligência da Clyvo.

É crucial configurar corretamente a chave da API (`GEMINI_API_KEY`) para garantir que o assistente possa funcionar sem interrupções.

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Cliente criado com sucesso.")

## 2. 📝 System Prompt: A Personalidade da Clyvo

O System Prompt é o coração da Clyvo! Ele define a persona do assistente, suas regras de conduta, o escopo de atuação e, o mais importante, os *guardrails* que garantem que a Clyvo se mantenha focada em saúde pet e serviços da Clyvo.

Leia atentamente para entender como a Clyvo foi instruída a interagir com os tutores e quais são suas prioridades no atendimento.

In [ ]:
MODEL = "gemini-3-flash-preview"
SYSTEM_PROMPT = """
# CONTEXTO E PERSONA
Você é o assistente virtual oficial da **Clyvo**, uma plataforma de e-commerce e planos de saúde pet.
Seu objetivo é orientar tutores na escolha do melhor plano de saúde para o pet deles e localizar clínicas parceiras credenciadas.

# GUARDRAIL DE ESCOPO (RIGOROSO)
Você é um robô de atendimento EXCLUSIVO da Clyvo.
Se o cliente fizer QUALQUER pergunta ou pedido fora do contexto de planos de saúde pet ou clínicas credenciadas (ex: ensinar programação Python, falar sobre tempo, curiosidades, piadas, futebol):
1. Recuse imediatamente e de forma cortês.
2. Informe que você foi programado APENAS para ajudar com planos de saúde e serviços da Clyvo.
3. Traga o foco de volta perguntando sobre as necessidades de saúde do pet dele.

# FLUXO DE ATENDIMENTO E FERRAMENTAS
1. Para recomendar o plano ideal, colete de forma natural:
   - Idade do pet
   - Se possui doença crônica ou tratamento contínuo (True/False)
   - Orçamento mensal máximo do tutor (R$)
   - Nível de tolerância a risco / cobertura desejada (baixo, medio, medio-alto, alto)
   Em seguida, chame a ferramenta `recomendar_plano`.

2. Caso o cliente solicite clínicas ou hospitais próximos, peça a localização geográfica do tutor e chame a ferramenta `buscar_clinicas_proximas`.

# REGRA DE CONFIRMAÇÃO E FICHA FINAL
Nunca confirme sem validação prévia. Ao definir o plano com o tutor:
1. Apresente o resumo completo (Nome do tutor, Nome do pet, Plano escolhido e Valor mensal).
2. Pergunte EXPLICITAMENTE: "Todos os dados estão corretos? Posso confirmar a adesão do seu plano?"
3. Apenas após a confirmação do cliente, finalize o atendimento.
"""

### 3. 🛠️ Ferramentas da Clyvo (Tools Callings)

Aqui, definimos as funcionalidades extras que a Clyvo pode acionar para auxiliar os tutores. Estas "ferramentas" são funções Python que o modelo de linguagem pode "chamar" quando identifica a intenção do usuário.

- **`recomendar_plano`:** Uma função inteligente para analisar as necessidades do pet (idade, doenças, orçamento) e sugerir o melhor plano de saúde.
- **`buscar_clinicas_proximas`:** Permite encontrar clínicas veterinárias parceiras com base na localização do tutor e no plano do pet.

Explore as implementações detalhadas de cada ferramenta para entender como elas contribuem para a eficácia da Clyvo!


In [ ]:
import math

# ==========================================
# 1. BASE DE DADOS E NOVAS DEFS DA CLYVO
# ==========================================

PLANOS = [
    {
        "id": "essencial",
        "nome": "Clyvo Essencial",
        "preco": 49.90,
        "badge": "Entrada",
        "perfil_ideal": {"idade_max": 15, "risco": "baixo"},
        "coberturas": ["Vacinas anuais", "Consultas de rotina", "Exames laboratoriais básicos"],
        "desconto_parceiros": 0.10
    },
    {
        "id": "conforto",
        "nome": "Clyvo Conforto",
        "preco": 89.90,
        "badge": "Mais Popular",
        "perfil_ideal": {"idade_max": 12, "risco": "medio"},
        "coberturas": ["Consultas ilimitadas", "Exames de imagem", "Vacinas", "Emergência 24h"],
        "desconto_parceiros": 0.20
    },
    {
        "id": "plus",
        "nome": "Clyvo Plus",
        "preco": 149.90,
        "badge": "Completo",
        "perfil_ideal": {"idade_max": 10, "risco": "medio-alto"},
        "coberturas": ["Cirurgias", "Internação", "Especialistas", "Exames complexos"],
        "desconto_parceiros": 0.30
    },
    {
        "id": "senior",
        "nome": "Clyvo Sênior & Especial",
        "preco": 239.90,
        "badge": "Especializado",
        "perfil_ideal": {"idade_max": 99, "risco": "alto"},
        "coberturas": ["Fisioterapia", "Acompanhamento oncologico/geriatra", "UTI", "Medicamentos contínuos"],
        "desconto_parceiros": 0.40
    }
]

CLINICAS_PARCEIRAS = [
    {
        "nome": "VetCare 24h Central",
        "lat": -23.561684,
        "lon": -46.655981,
        "emergencia_24h": True,
        "especialidades": ["Clínica Geral", "Cirurgia", "UTI", "Geriatria"],
        "planos_aceitos": ["essencial", "conforto", "plus", "senior"]
    },
    {
        "nome": "Clínica Veterinária PetLife",
        "lat": -23.570000,
        "lon": -46.640000,
        "emergencia_24h": False,
        "especialidades": ["Clínica Geral", "Vacinação", "Dermatologia"],
        "planos_aceitos": ["conforto", "plus", "senior"]
    },
    {
        "nome": "Hospital Especializado Animal Primus",
        "lat": -23.540000,
        "lon": -46.630000,
        "emergencia_24h": True,
        "especialidades": ["Oncologia", "Fisioterapia", "Ortopedia", "UTI"],
        "planos_aceitos": ["plus", "senior"]
    }
]


def recomendar_plano(idade_pet: int, tem_doenca_cronica: bool, orcamento_mensal: float, tolerancia_risco: str) -> dict:
    """Calcula o melhor plano de saúde pet com base no orçamento e perfil do animal."""
    tolerancia_risco = tolerancia_risco.lower().strip()
    ordem_risco = ["baixo", "medio", "medio-alto", "alto"]

    if tem_doenca_cronica:
        plano_senior = next(p for p in PLANOS if p["id"] == "senior")
        if orcamento_mensal >= plano_senior["preco"]:
            return {
                "plano_recomendado": plano_senior,
                "justificativa": "Pet com doença crônica necessita de acompanhamento contínuo no plano Sênior & Especial."
            }
        else:
            candidatos_orcamento = [p for p in PLANOS if p["preco"] <= orcamento_mensal]
            if candidatos_orcamento:
                melhor = max(candidatos_orcamento, key=lambda p: p["preco"])
                return {
                    "plano_recomendado": melhor,
                    "justificativa": f"O plano ideal é o Sênior & Especial, mas foi indicado o {melhor['nome']} para respeitar o orçamento de R$ {orcamento_mensal:.2f}."
                }

    candidatos = []
    for p in PLANOS:
        perfil = p["perfil_ideal"]
        if p["id"] == "senior" and not tem_doenca_cronica and idade_pet < 8:
            continue
        if idade_pet > perfil["idade_max"] or orcamento_mensal < p["preco"]:
            continue
        if ordem_risco.index(tolerancia_risco) < ordem_risco.index(perfil["risco"]):
            continue
        candidatos.append(p)

    if not candidatos:
        return {
            "plano_recomendado": PLANOS[0],
            "justificativa": "Nenhum plano atendeu 100% dos filtros. Sugerido Clyvo Essencial como opção de entrada."
        }

    melhor_plano = max(candidatos, key=lambda item: item["preco"])
    return {
        "plano_recomendado": melhor_plano,
        "justificativa": "Atende totalmente à idade, nível de risco e orçamento do tutor."
    }


def distancia_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    return R * (2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)))


def buscar_clinicas_proximas(lat_usuario: float, lon_usuario: float, plano_id: str, raio_max_km: float = 15.0) -> dict:
    """Busca clínicas parceiras credenciadas e verifica cobertura do plano do tutor."""
    resultados = []
    for clinica in CLINICAS_PARCEIRAS:
        dist = distancia_km(lat_usuario, lon_usuario, clinica["lat"], clinica["lon"])
        if dist <= raio_max_km:
            coberto = plano_id in clinica["planos_aceitos"]
            resultados.append({
                "nome": clinica["nome"],
                "distancia_km": round(dist, 2),
                "emergencia_24h": clinica["emergencia_24h"],
                "especialidades": clinica["especialidades"],
                "status_cobertura": "Atendimento Coberto" if coberto else "Atendimento com Desconto (Co-participação)"
            })
    resultados.sort(key=lambda x: x["distancia_km"])
    return {"clinicas": resultados}


# ==========================================
# 2. DECLARAÇÃO DAS FERRAMENTAS (SCHEMA GEMINI)
# ==========================================

FERRAMENTAS = [
    {
        "type": "function",
        "name": "recomendar_plano",
        "description": "Calcula e recomenda o plano de saúde pet ideal da Clyvo com base na idade, histórico de doença, orçamento e risco.",
        "parameters": {
            "type": "object",
            "properties": {
                "idade_pet": {"type": "integer", "description": "Idade do animal em anos."},
                "tem_doenca_cronica": {"type": "boolean", "description": "Se o pet possui doença crônica ou tratamento diário."},
                "orcamento_mensal": {"type": "number", "description": "Orçamento máximo mensal em R$ que o tutor aceita pagar."},
                "tolerancia_risco": {
                    "type": "string",
                    "enum": ["baixo", "medio", "medio-alto", "alto"],
                    "description": "Perfil de tolerância a risco do tutor."
                }
            },
            "required": ["idade_pet", "tem_doenca_cronica", "orcamento_mensal", "tolerancia_risco"],
        },
    },
    {
        "type": "function",
        "name": "buscar_clinicas_proximas",
        "description": "Busca as clínicas credenciadas mais próximas da localização do tutor e confere a cobertura do plano.",
        "parameters": {
            "type": "object",
            "properties": {
                "lat_usuario": {"type": "number", "description": "Latitude da posição atual do tutor."},
                "lon_usuario": {"type": "number", "description": "Longitude da posição atual do tutor."},
                "plano_id": {
                    "type": "string",
                    "enum": ["essencial", "conforto", "plus", "senior"],
                    "description": "ID do plano contratado ou analisado."
                },
                "raio_max_km": {"type": "number", "description": "Raio máximo de busca em km."}
            },
            "required": ["lat_usuario", "lon_usuario", "plano_id"],
        },
    },
]

FUNCOES_AUTORIZADAS = {
    "recomendar_plano": recomendar_plano,
    "buscar_clinicas_proximas": buscar_clinicas_proximas,
}

## 4. 📝 Geração de Ficha e Validação com Pydantic

O coração da **Clyvo** na parte de contratação! Nesta seção, definimos a estrutura `ContratacaoPlanoClyvo` usando a poderosa biblioteca [Pydantic](https://docs.pydantic.dev/latest/), que nos permite:

- **Validar Dados:** Garantir que as informações extraídas do plano estejam sempre corretas e no formato esperado.
- **Extrair Estruturas:** Transformar o histórico da conversa em um objeto JSON tipado, facilitando o processamento e armazenamento.

A função `gerar_ficha_pydantic` utiliza um prompt específico para guiar o modelo Gemini na análise do diálogo e preencher automaticamente uma instância da nossa classe. Isso assegura que dados cruciais como nome do tutor, nome do pet, idade, plano escolhido e valor mensal sejam capturados com precisão.

In [ ]:
from pydantic import BaseModel
from typing import Literal
from google.genai import types

# Estrutura Pydantic para validar e extrair a contratação do plano
class ContratacaoPlanoClyvo(BaseModel):
    nome_tutor: str
    nome_pet: str
    especie: Literal["cao", "gato"]
    idade_pet: int
    tem_doenca_cronica: bool
    plano_escolhido: str
    valor_mensal: float

def gerar_ficha_pydantic(historico_conversa: str) -> ContratacaoPlanoClyvo:
    prompt_extracao = f"""
    Você é um extrator de dados.
    Analise o histórico de conversa do atendimento da Clyvo abaixo e extraia com precisão os dados do plano de saúde pet contratado e confirmado.

    HISTÓRICO DA CONVERSA:
    {historico_conversa}
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt_extracao,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ContratacaoPlanoClyvo,
        ),
    )

    return ContratacaoPlanoClyvo.model_validate_json(response.text)

### 5. 💬 Chat

A seção **Chat** é o palco principal onde a mágica acontece! ✨

Aqui, implementamos a lógica do nosso assistente virtual, criando um loop interativo onde o tutor pode conversar livremente com a Clyvo. Veja como funciona:

- **Histórico Contínuo:** Mantemos um `historico_texto` completo da conversa, garantindo que a Clyvo sempre entenda o contexto.
- **Inteligência Ativa:** Para cada mensagem do usuário, o modelo Gemini é acionado, utilizando o `SYSTEM_PROMPT` para manter a persona da Clyvo e as `FERRAMENTAS` disponíveis.
- **Ações Inteligentes:** Se a Clyvo identifica uma intenção (como "recomendar plano" ou "buscar clínica"), ela automaticamente chama a função Python correspondente! O resultado é processado e enviado de volta ao modelo para gerar uma resposta coerente e útil.

Isso permite que a Clyvo não apenas converse, mas também execute ações e resolva as necessidades do tutor de forma dinâmica e eficiente!

In [ ]:
import json

previous_interaction_id = None
historico_texto = []

print("Olá! Sou a assistente virtual da Clyvo.")
print("Digite 'sair' para encerrar.\n")

while True:
    mensagem = input("Você: ").strip()

    if mensagem.lower() in ["sair", "fim", "encerrar"]:
        print("Clyvo: Atendimento encerrado. Até mais!")
        break

    if not mensagem:
        continue

    historico_texto.append(f"Cliente: {mensagem}")

    argumentos = {
        "model": MODEL,
        "system_instruction": SYSTEM_PROMPT,
        "input": mensagem,
        "tools": FERRAMENTAS,
    }

    if previous_interaction_id is not None:
        argumentos["previous_interaction_id"] = previous_interaction_id

    interaction = client.interactions.create(**argumentos)
    previous_interaction_id = interaction.id

    chamadas = [etapa for etapa in interaction.steps if etapa.type == "function_call"]

    if chamadas:
        function_results = []
        for chamada in chamadas:
            nome_funcao = chamada.name
            args = chamada.arguments

            if nome_funcao in FUNCOES_AUTORIZADAS:
                resultado = FUNCOES_AUTORIZADAS[nome_funcao](**args)
            else:
                resultado = {"erro": "Função não autorizada."}

            function_results.append({
                "type": "function_result",
                "name": nome_funcao,
                "call_id": chamada.id,
                "result": [{"type": "text", "text": json.dumps(resultado, ensure_ascii=False)}],
            })

        interaction_final = client.interactions.create(
            model=MODEL,
            system_instruction=SYSTEM_PROMPT,
            input=function_results,
            tools=FERRAMENTAS,
            previous_interaction_id=previous_interaction_id,
        )
        previous_interaction_id = interaction_final.id
        resposta_clyvo = interaction_final.output_text
    else:
        resposta_clyvo = interaction.output_text

    historico_texto.append(f"Clyvo: {resposta_clyvo}")
    print("Clyvo:", resposta_clyvo)
    print()

### 6. 📄 Gerar Ficha Final

Chegamos à etapa crucial de **finalização**! 🎉

Nesta seção, o `historico_texto` completo de toda a conversa é utilizado para criar a tão esperada **ficha final de contratação**.

A mágica acontece quando invocamos a função `gerar_ficha_pydantic` (que definimos anteriormente). Ela analisa todo o diálogo, extrai os detalhes essenciais do plano escolhido e do pet, e os organiza em uma estrutura clara e validada pelo Pydantic.

O resultado é uma ficha completa, exibida tanto como um objeto Python quanto em um formato JSON limpo e legível. Isso demonstra o poder de transformar interações conversacionais em dados estruturados e prontos para uso!

In [ ]:
# Junta todo o histórico em um texto único
conversa_completa = "\n".join(historico_texto)

# Gera a ficha validada via Pydantic
ficha_final = gerar_ficha_pydantic(conversa_completa)

# Exibe a ficha gerada
print("=== FICHA FINAL DE CONTRATAÇÃO (PYDANTIC) ===")
print(ficha_final)

# Exibe em formato JSON formatado:
print("\nJSON Formatado:")
print(ficha_final.model_dump_json(indent=2))